In [1]:
# MQ-019: MedGemma 4B IT — Repository Access Verification

from huggingface_hub import HfApi

MODEL_ID = "google/medgemma-4b-it"

api = HfApi()
files = api.list_repo_files(MODEL_ID)

print("Repository access: SUCCESS")
print("Model:", MODEL_ID)
print("Total files:", len(files))
print("Weight files:", [
    name for name in files
    if name.endswith(".safetensors")
])

Repository access: SUCCESS
Model: google/medgemma-4b-it
Total files: 15
Weight files: ['model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']


In [2]:
# MQ-019: SageMaker hardware assessment

import os
import shutil
import torch

# CPU and RAM
print("CPU cores:", os.cpu_count())

with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal:"):
            ram_gb = int(line.split()[1]) / 1024**2
            print(f"Total RAM: {ram_gb:.2f} GB")
            break

# Available disk space
disk = shutil.disk_usage("/home/sagemaker-user")

print(f"Free disk space: {disk.free / 1024**3:.2f} GB")

# GPU availability
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("No CUDA GPU detected.")

CPU cores: 2
Total RAM: 3.74 GB
Free disk space: 14.04 GB
CUDA available: False
No CUDA GPU detected.


In [3]:
import boto3

session = boto3.Session()
print("AWS Region:", session.region_name)

AWS Region: us-east-1


In [4]:
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name="us-east-1")

try:
    response = sm.list_training_jobs(MaxResults=5)

    print("SageMaker training API access: SUCCESS")
    print("Existing training jobs:", len(response["TrainingJobSummaries"]))

except ClientError as e:
    print("SageMaker training API access: FAILED")
    print("Error code:", e.response["Error"]["Code"])
    print("Message:", e.response["Error"]["Message"])

SageMaker training API access: SUCCESS
Existing training jobs: 1


In [5]:
import boto3
import pandas as pd

sm = boto3.client("sagemaker", region_name="us-east-1")

response = sm.list_training_jobs(
    MaxResults=20,
    SortBy="CreationTime",
    SortOrder="Descending"
)

jobs = [
    {
        "JobName": job["TrainingJobName"],
        "Status": job["TrainingJobStatus"],
        "Created": job["CreationTime"]
    }
    for job in response["TrainingJobSummaries"]
]

display(pd.DataFrame(jobs))

,JobName,Status,Created
0,sagemaker-xgboost-2024-04-23-00-17-17-973,Completed,2024-04-23 00:17:18.134000+00:00


In [6]:
import boto3

sts = boto3.client("sts")

identity = sts.get_caller_identity()

print("AWS Account:", identity["Account"])
print("IAM Identity:", identity["Arn"])

AWS Account: 945737411191
IAM Identity: arn:aws:sts::945737411191:assumed-role/LabRole/SageMaker


In [7]:
import boto3
from botocore.exceptions import ClientError

iam = boto3.client("iam", region_name="us-east-1")

role_arn = "arn:aws:iam::945737411191:role/LabRole"

try:
    response = iam.simulate_principal_policy(
        PolicySourceArn=role_arn,
        ActionNames=[
            "sagemaker:CreateTrainingJob",
            "iam:PassRole"
        ]
    )

    for result in response["EvaluationResults"]:
        print(
            result["EvalActionName"],
            "=>",
            result["EvalDecision"]
        )

except ClientError as e:
    print("IAM simulation unavailable")
    print("Error:", e.response["Error"]["Code"])

sagemaker:CreateTrainingJob => allowed
iam:PassRole => implicitDeny


In [8]:
import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name="us-east-1")
iam = boto3.client("iam", region_name="us-east-1")

try:
    # Retrieve the execution role of the existing SageMaker environment
    execution_role = "arn:aws:iam::945737411191:role/LabRole"

    response = iam.simulate_principal_policy(
        PolicySourceArn=execution_role,
        ActionNames=["iam:PassRole"],
        ResourceArns=[execution_role],
        ContextEntries=[
            {
                "ContextKeyName": "iam:PassedToService",
                "ContextKeyValues": ["sagemaker.amazonaws.com"],
                "ContextKeyType": "string"
            }
        ]
    )

    for result in response["EvaluationResults"]:
        print("Action:", result["EvalActionName"])
        print("Decision:", result["EvalDecision"])

except ClientError as e:
    print("Permission check failed:", e.response["Error"]["Code"])

Action: iam:PassRole
Decision: allowed


In [9]:
import boto3
from botocore.exceptions import ClientError

pricing = boto3.client("pricing", region_name="us-east-1")

try:
    response = pricing.describe_services(
        ServiceCode="AmazonSageMaker",
        MaxResults=1
    )

    print("AWS Pricing API: ACCESSIBLE")
    print("Service:", response["Services"][0]["ServiceCode"])

except ClientError as e:
    print("AWS Pricing API: UNAVAILABLE")
    print("Error:", e.response["Error"]["Code"])

AWS Pricing API: UNAVAILABLE
Error: AccessDeniedException


In [10]:
# MQ-019: MedGemma inference configuration

MODEL_ID = "google/medgemma-4b-it"
AWS_REGION = "us-east-1"
GPU_INSTANCE = "ml.g5.xlarge"

# Limit generation during initial evaluation
MAX_NEW_TOKENS = 256

# Research settings
TEMPERATURE = 0.0
TOP_K_PASSAGES = 5

print("Model:", MODEL_ID)
print("Region:", AWS_REGION)
print("Planned instance:", GPU_INSTANCE)
print("Top-K passages:", TOP_K_PASSAGES)
print("Maximum generated tokens:", MAX_NEW_TOKENS)
print("Configuration ready. No GPU launched.")

Model: google/medgemma-4b-it
Region: us-east-1
Planned instance: ml.g5.xlarge
Top-K passages: 5
Maximum generated tokens: 256
Configuration ready. No GPU launched.


In [11]:
# MQ-019: Evidence-grounded medical RAG prompt

def build_medgemma_prompt(question, retrieved_passages):
    """
    Construct a medical question-answering prompt
    using passages retrieved from MedQuAD.

    retrieved_passages: DataFrame containing
    'passage' and 'source' columns.
    """

    evidence_blocks = []

    for i, (_, row) in enumerate(
        retrieved_passages.iterrows(), start=1
    ):
        evidence_blocks.append(
            f"[{i}] Source: {row['source']}\n"
            f"Evidence: {row['passage']}"
        )

    evidence_text = "\n\n".join(evidence_blocks)

    prompt = f"""
You are MedQuad AI, a medical information assistant.

Answer the user's medical question using only
the evidence provided below.

Rules:
1. Do not invent medical facts.
2. Do not provide a diagnosis or prescribe medication.
3. If the evidence is insufficient, clearly state
   that the available information is insufficient.
4. Do not confuse similarly named medical conditions.
5. Preserve uncertainty expressed in the evidence.
6. Cite supporting evidence using [1], [2], etc.
7. Provide a clear, concise answer.

Medical evidence:
{evidence_text}

User question:
{question}

Answer:
"""

    return prompt.strip()


print("MedGemma RAG prompt builder ready.")

MedGemma RAG prompt builder ready.


In [12]:
from pathlib import Path
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path("/home/sagemaker-user/medical_chatbot")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load existing medical passages
passages_df = pd.read_csv(
    PROCESSED_DIR / "medquad_final_passages.csv"
)

# Load existing FAISS index
faiss_index = faiss.read_index(
    str(PROCESSED_DIR / "medquad_faiss.index")
)

# Load the same embedding model used to build the index
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

assert faiss_index.ntotal == len(passages_df)
assert faiss_index.d == 384

def retrieve_medical_passages(query, top_k=5):
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu"
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vector, top_k)

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        if idx < 0:
            continue

        record = passages_df.iloc[int(idx)]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "question": record["question"],
            "passage": record["passage"],
            "source": record["source"],
            "focus_area": record["focus_area"]
        })

    return pd.DataFrame(results)

print("FAISS index loaded:", faiss_index.ntotal, "passages")
print("Retrieval pipeline ready.")

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1490.83it/s]


FAISS index loaded: 38544 passages
Retrieval pipeline ready.


In [13]:
test_question = "What causes Hanhart syndrome?"

retrieved = retrieve_medical_passages(
    test_question,
    top_k=TOP_K_PASSAGES
)

test_prompt = build_medgemma_prompt(
    test_question,
    retrieved
)

print("Retrieved passages:", len(retrieved))
print("Prompt length:", len(test_prompt), "characters")
print("\n", test_prompt[:3000])

Retrieved passages: 5
Prompt length: 3637 characters

 You are MedQuad AI, a medical information assistant.

Answer the user's medical question using only
the evidence provided below.

Rules:
1. Do not invent medical facts.
2. Do not provide a diagnosis or prescribe medication.
3. If the evidence is insufficient, clearly state
   that the available information is insufficient.
4. Do not confuse similarly named medical conditions.
5. Preserve uncertainty expressed in the evidence.
6. Cite supporting evidence using [1], [2], etc.
7. Provide a clear, concise answer.

Medical evidence:
[1] Source: GARD
Evidence: Hanhart syndrome is a rare condition that primarily affects the craniofacial region and the limbs (arms and legs). People affected by this condition are often born with a short, incompletely developed tongue; absent or partially missing fingers and/or toes; abnormalities of the arms and/or legs; and an extremely small jaw. The severity of these physical abnormalities varies greatly

In [14]:
# MQ-019: Prototype evidence filter for cause-related questions

def filter_cause_evidence(retrieved_df):
    """
    Retain passages containing explicit causal language.
    Intended only for initial cause-related question testing.
    """

    causal_phrases = [
        "cause of",
        "caused by",
        "causes of",
        "cause is",
        "cause remains",
        "etiology",
        "aetiology"
    ]

    filtered = retrieved_df[
        retrieved_df["passage"].fillna("").str.lower().apply(
            lambda text: any(
                phrase in text for phrase in causal_phrases
            )
        )
    ].copy()

    return filtered.reset_index(drop=True)


filtered_retrieved = filter_cause_evidence(retrieved)

print("Original passages:", len(retrieved))
print("Filtered passages:", len(filtered_retrieved))

display(
    filtered_retrieved[
        ["rank", "similarity", "focus_area", "passage"]
    ]
)

Original passages: 5
Filtered passages: 1


,rank,similarity,focus_area,passage
0,1,0.75766,Hanhart syndrome,Hanhart syndrome is a rare condition that prim...


In [15]:
# MQ-019: Build filtered, evidence-grounded prompt

if filtered_retrieved.empty:
    print("No matching evidence found.")
    print("MedGemma generation should be skipped.")
else:
    filtered_prompt = build_medgemma_prompt(
        question=test_question,
        retrieved_passages=filtered_retrieved
    )

    print("Question:", test_question)
    print("Evidence passages:", len(filtered_retrieved))
    print("Prompt characters:", len(filtered_prompt))
    print("\n--- FINAL MEDGEMMA PROMPT ---\n")
    print(filtered_prompt)

Question: What causes Hanhart syndrome?
Evidence passages: 1
Prompt characters: 1226

--- FINAL MEDGEMMA PROMPT ---

You are MedQuad AI, a medical information assistant.

Answer the user's medical question using only
the evidence provided below.

Rules:
1. Do not invent medical facts.
2. Do not provide a diagnosis or prescribe medication.
3. If the evidence is insufficient, clearly state
   that the available information is insufficient.
4. Do not confuse similarly named medical conditions.
5. Preserve uncertainty expressed in the evidence.
6. Cite supporting evidence using [1], [2], etc.
7. Provide a clear, concise answer.

Medical evidence:
[1] Source: GARD
Evidence: Hanhart syndrome is a rare condition that primarily affects the craniofacial region and the limbs (arms and legs). People affected by this condition are often born with a short, incompletely developed tongue; absent or partially missing fingers and/or toes; abnormalities of the arms and/or legs; and an extremely small ja

In [16]:
# MQ-019: MedGemma inference function
# Define only — do not load the model on this CPU instance.

def generate_medgemma_answer(
    prompt,
    model,
    processor,
    max_new_tokens=256
):
    import torch

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    # Remove the input prompt tokens
    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    answer = processor.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer.strip()


print("MedGemma inference function defined.")
print("Model not loaded. No GPU launched.")

MedGemma inference function defined.
Model not loaded. No GPU launched.


In [17]:
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/medical_chatbot")

EVALUATION_DIR = (
    PROJECT_ROOT / "data" / "processed" / "medgemma_evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prompt_path = EVALUATION_DIR / "hanhart_filtered_prompt.txt"

assert not filtered_retrieved.empty, (
    "No relevant evidence available. "
    "Cannot create evaluation prompt."
)

filtered_prompt = build_medgemma_prompt(
    question=test_question,
    retrieved_passages=filtered_retrieved
)

prompt_path.write_text(
    filtered_prompt,
    encoding="utf-8"
)

print("Evaluation prompt saved:", prompt_path)
print("Prompt characters:", len(filtered_prompt))
print("Evidence passages:", len(filtered_retrieved))
print("File exists:", prompt_path.exists())

Evaluation prompt saved: /home/sagemaker-user/medical_chatbot/data/processed/medgemma_evaluation/hanhart_filtered_prompt.txt
Prompt characters: 1226
Evidence passages: 1
File exists: True


In [18]:
# MQ-019: Save baseline RAG prompt

baseline_prompt = build_medgemma_prompt(
    question=test_question,
    retrieved_passages=retrieved
)

baseline_path = (
    EVALUATION_DIR / "hanhart_baseline_prompt.txt"
)

baseline_path.write_text(
    baseline_prompt,
    encoding="utf-8"
)

print("Baseline prompt saved:", baseline_path)
print("Baseline evidence passages:", len(retrieved))
print("Baseline prompt characters:", len(baseline_prompt))
print("File exists:", baseline_path.exists())

print("\n--- EXPERIMENT COMPARISON ---")
print("Baseline RAG:", len(retrieved), "passages")
print("Filtered RAG:", len(filtered_retrieved), "passages")

Baseline prompt saved: /home/sagemaker-user/medical_chatbot/data/processed/medgemma_evaluation/hanhart_baseline_prompt.txt
Baseline evidence passages: 5
Baseline prompt characters: 3637
File exists: True

--- EXPERIMENT COMPARISON ---
Baseline RAG: 5 passages
Filtered RAG: 1 passages


In [19]:
import json

experiment_config = {
    "experiment_id": "MQ-019",
    "model_id": MODEL_ID,
    "question": test_question,
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False
    },
    "experiments": [
        {
            "name": "baseline_rag",
            "prompt_file": "hanhart_baseline_prompt.txt",
            "evidence_count": len(retrieved),
            "output_file": "hanhart_baseline_answer.json"
        },
        {
            "name": "filtered_rag",
            "prompt_file": "hanhart_filtered_prompt.txt",
            "evidence_count": len(filtered_retrieved),
            "output_file": "hanhart_filtered_answer.json"
        }
    ]
}

config_path = EVALUATION_DIR / "experiment_config.json"

config_path.write_text(
    json.dumps(experiment_config, indent=2),
    encoding="utf-8"
)

# Verify the configuration and referenced prompts
saved_config = json.loads(
    config_path.read_text(encoding="utf-8")
)

for experiment in saved_config["experiments"]:
    prompt_file = EVALUATION_DIR / experiment["prompt_file"]
    assert prompt_file.is_file(), f"Missing: {prompt_file}"

print("Experiment configuration saved:", config_path)
print("Model:", saved_config["model_id"])

for experiment in saved_config["experiments"]:
    print(
        experiment["name"],
        "| Evidence:", experiment["evidence_count"],
        "| Prompt:", experiment["prompt_file"]
    )

print("All experiment inputs verified.")

Experiment configuration saved: /home/sagemaker-user/medical_chatbot/data/processed/medgemma_evaluation/experiment_config.json
Model: google/medgemma-4b-it
baseline_rag | Evidence: 5 | Prompt: hanhart_baseline_prompt.txt
filtered_rag | Evidence: 1 | Prompt: hanhart_filtered_prompt.txt
All experiment inputs verified.


In [20]:
import boto3
from botocore.exceptions import ClientError

s3 = boto3.client("s3", region_name="us-east-1")

BUCKET = "medquad-ai-cleaned"

INPUT_PREFIX = "medgemma/experiments/mq019/input/"
OUTPUT_PREFIX = "medgemma/experiments/mq019/output/"

try:
    s3.head_bucket(Bucket=BUCKET)

    print("S3 bucket accessible:", BUCKET)
    print("Input S3 URI:", f"s3://{BUCKET}/{INPUT_PREFIX}")
    print("Output S3 URI:", f"s3://{BUCKET}/{OUTPUT_PREFIX}")
    print("S3 location verification: PASSED")

except ClientError as error:
    print("S3 location verification: FAILED")
    print("Error:", error.response["Error"]["Code"])

S3 bucket accessible: medquad-ai-cleaned
Input S3 URI: s3://medquad-ai-cleaned/medgemma/experiments/mq019/input/
Output S3 URI: s3://medquad-ai-cleaned/medgemma/experiments/mq019/output/
S3 location verification: PASSED


In [21]:
from pathlib import Path
import boto3

s3 = boto3.client("s3", region_name="us-east-1")

BUCKET = "medquad-ai-cleaned"
INPUT_PREFIX = "medgemma/experiments/mq019/input/"

local_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/medgemma_evaluation"
)

files_to_upload = [
    "experiment_config.json",
    "hanhart_baseline_prompt.txt",
    "hanhart_filtered_prompt.txt",
]

for filename in files_to_upload:
    local_path = local_dir / filename

    assert local_path.is_file(), (
        f"Missing local file: {local_path}"
    )

    s3_key = INPUT_PREFIX + filename

    s3.upload_file(
        str(local_path),
        BUCKET,
        s3_key
    )

    # Confirm the uploaded object exists
    s3.head_object(
        Bucket=BUCKET,
        Key=s3_key
    )

    print(f"Uploaded and verified: {filename}")

print("\nAll MedGemma experiment inputs uploaded.")
print("S3 location:", f"s3://{BUCKET}/{INPUT_PREFIX}")

Uploaded and verified: experiment_config.json
Uploaded and verified: hanhart_baseline_prompt.txt
Uploaded and verified: hanhart_filtered_prompt.txt

All MedGemma experiment inputs uploaded.
S3 location: s3://medquad-ai-cleaned/medgemma/experiments/mq019/input/


In [22]:
from pathlib import Path
import boto3

s3 = boto3.client("s3", region_name="us-east-1")

bucket = "medquad-ai-cleaned"
key = "medgemma/experiments/mq019/code/medgemma_source.tar.gz"

archive_path = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/medgemma_evaluation/"
    "medgemma_source.tar.gz"
)

assert archive_path.is_file(), "Execution package missing"

s3.upload_file(
    str(archive_path),
    bucket,
    key
)

response = s3.head_object(
    Bucket=bucket,
    Key=key
)

print("Execution package uploaded successfully.")
print("S3 URI:", f"s3://{bucket}/{key}")
print("Uploaded size:", response["ContentLength"], "bytes")
print("No GPU launched.")

Execution package uploaded successfully.
S3 URI: s3://medquad-ai-cleaned/medgemma/experiments/mq019/code/medgemma_source.tar.gz
Uploaded size: 2199 bytes
No GPU launched.


In [23]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session(region_name="us-east-1")

sts = session.client("sts")
secrets = session.client("secretsmanager")

print("AWS region:", session.region_name)
print("AWS identity:", sts.get_caller_identity()["Arn"])

try:
    response = secrets.list_secrets(MaxResults=1)

    print("Secrets Manager: ACCESSIBLE")
    print("Secret listing permitted.")

except ClientError as error:
    print("Secrets Manager: RESTRICTED")
    print("Error:", error.response["Error"]["Code"])

print("No secrets created or retrieved.")
print("No Hugging Face token displayed.")
print("No GPU launched.")

AWS region: us-east-1
AWS identity: arn:aws:sts::945737411191:assumed-role/LabRole/SageMaker
Secrets Manager: ACCESSIBLE
Secret listing permitted.
No secrets created or retrieved.
No Hugging Face token displayed.
No GPU launched.


In [24]:
import boto3
from botocore.exceptions import ClientError

iam = boto3.client("iam", region_name="us-east-1")

role_arn = "arn:aws:iam::945737411191:role/LabRole"

secret_arn = (
    "arn:aws:secretsmanager:us-east-1:"
    "945737411191:secret:medquad/huggingface-*"
)

try:
    response = iam.simulate_principal_policy(
        PolicySourceArn=role_arn,
        ActionNames=[
            "secretsmanager:CreateSecret",
            "secretsmanager:GetSecretValue"
        ],
        ResourceArns=[secret_arn]
    )

    for result in response["EvaluationResults"]:
        print(
            result["EvalActionName"],
            "=>",
            result["EvalDecision"]
        )

except ClientError as error:
    print(
        "Permission simulation unavailable:",
        error.response["Error"]["Code"]
    )

print("\nNo secrets created or retrieved.")
print("No Hugging Face token displayed.")
print("No GPU launched.")

secretsmanager:CreateSecret => allowed
secretsmanager:GetSecretValue => allowed

No secrets created or retrieved.
No Hugging Face token displayed.
No GPU launched.


In [25]:
import boto3
from botocore.exceptions import ClientError

secrets = boto3.client(
    "secretsmanager",
    region_name="us-east-1"
)

SECRET_NAME = "medquad/huggingface-token"

try:
    response = secrets.get_secret_value(
        SecretId=SECRET_NAME
    )

    token = response.get("SecretString", "")

    if token.strip():
        print("Secret retrieval: SUCCESS")
        print("Token value: NOT DISPLAYED")
        print("Secret is nonempty: True")
    else:
        print("Secret retrieval: FAILED — empty value")

    del token
    del response

except ClientError as error:
    print("Secret retrieval: FAILED")
    print("Error:", error.response["Error"]["Code"])

print("No GPU launched.")

Secret retrieval: SUCCESS
Token value: NOT DISPLAYED
Secret is nonempty: True
No GPU launched.


In [26]:
import importlib.metadata as metadata
import torch

packages = [
    "sagemaker",
    "torch",
    "transformers",
    "accelerate",
    "boto3"
]

print("=== Environment Versions ===")

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n=== Hardware ===")
print("CUDA available:", torch.cuda.is_available())

print("\nNo GPU launched.")

=== Environment Versions ===
sagemaker: 2.257.7
torch: 2.14.1
transformers: 5.20.1
accelerate: 1.15.0
boto3: 1.43.114

=== Hardware ===
CUDA available: False

No GPU launched.


In [27]:
import importlib.util
import sys
import subprocess

print("Python executable:", sys.executable)

sdk_spec = importlib.util.find_spec("sagemaker")

print(
    "SageMaker SDK installed:",
    sdk_spec is not None
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "index",
        "versions",
        "sagemaker"
    ],
    capture_output=True,
    text=True
)

print("\nAvailable SageMaker SDK versions:")
print((result.stdout or result.stderr)[:1500])

print("\nNo packages installed.")
print("No GPU launched.")

Python executable: /home/sagemaker-user/medquad-envs/rag-env/bin/python
SageMaker SDK installed: True



Available SageMaker SDK versions:
sagemaker (3.25.0)
Available versions: 3.25.0, 3.24.0, 3.23.0, 3.22.1, 3.22.0, 3.21.0, 3.20.0, 3.19.0, 3.18.0, 3.17.0, 3.16.0, 3.15.1, 3.15.0, 3.14.0, 3.13.1, 3.13.0, 3.12.0, 3.11.0, 3.10.1, 3.10.0, 3.9.0, 3.8.0, 3.7.1, 3.7.0, 3.6.0, 3.5.0, 3.4.1, 3.4.0, 3.3.1, 3.3.0, 3.2.0, 3.1.1, 3.1.0, 3.0.1, 3.0, 2.257.7, 2.257.6, 2.257.5, 2.257.4, 2.257.3, 2.257.2, 2.257.1, 2.257.0, 2.256.1, 2.256.0, 2.255.0, 2.254.1, 2.254.0, 2.253.1, 2.252.0, 2.251.1, 2.251.0, 2.250.0, 2.249.0, 2.248.2, 2.248.1, 2.248.0, 2.247.1, 2.247.0, 2.246.0, 2.245.0, 2.244.2, 2.244.1, 2.244.0, 2.243.4, 2.243.3, 2.243.2, 2.243.1, 2.243.0, 2.242.0, 2.241.0, 2.240.0, 2.239.3, 2.239.2, 2.239.1, 2.239.0, 2.238.0, 2.237.3, 2.237.1, 2.237.0, 2.236.0, 2.235.2, 2.235.1, 2.235.0, 2.234.0, 2.233.0, 2.232.3, 2.232.2, 2.232.1, 2.232.0, 2.231.0, 2.230.0, 2.229.0, 2.228.0, 2.227.0, 2.226.1, 2.226.0, 2.225.0, 2.224.4, 2.224.3, 2.224.2, 2.224.1, 2.224.0, 2.223.0, 2.222.1, 2.222.0, 2.221.1, 2.221.0, 2.220.

In [28]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "sagemaker==2.257.7"
])

print("SageMaker SDK installation completed.")

SageMaker SDK installation completed.


In [29]:
import importlib.metadata

print(
    "SageMaker SDK:",
    importlib.metadata.version("sagemaker")
)

SageMaker SDK: 2.257.7


In [30]:
from sagemaker import image_uris

try:
    image_uri = image_uris.retrieve(
        framework="pytorch",
        region="us-east-1",
        version="2.5.1",
        py_version="py311",
        image_scope="training",
        instance_type="ml.g5.xlarge",
    )

    print("GPU container image resolved.")
    print("Image URI:", image_uri)
    print("Framework: PyTorch 2.5.1")
    print("Python: 3.11")
    print("Instance: ml.g5.xlarge")

except Exception as error:
    print("Container resolution failed.")
    print("Error type:", type(error).__name__)
    print("Details:", str(error)[:1000])

print("No training job created.")
print("No GPU launched.")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


GPU container image resolved.
Image URI: 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.5.1-gpu-py311
Framework: PyTorch 2.5.1
Python: 3.11
Instance: ml.g5.xlarge
No training job created.
No GPU launched.


In [31]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch

session = boto3.Session(region_name="us-east-1")

sm_session = sagemaker.Session(
    boto_session=session
)

role = "arn:aws:iam::945737411191:role/LabRole"

source_dir = (
    "/home/sagemaker-user/medical_chatbot/"
    "deployment/medgemma"
)

estimator = PyTorch(
    entry_point="medgemma_job.py",
    source_dir=source_dir,
    role=role,
    framework_version="2.5.1",
    py_version="py311",
    instance_count=1,
    instance_type="ml.g5.xlarge",
    volume_size=50,
    max_run=1800,
    output_path=(
        "s3://medquad-ai-cleaned/"
        "medgemma/experiments/mq019/output/"
    ),
    sagemaker_session=sm_session,
    disable_profiler=True,
    debugger_hook_config=False,
)

print("SageMaker Estimator: CONFIGURED")
print("Instance:", estimator.instance_type)
print("Instance count:", estimator.instance_count)
print("Maximum runtime:", estimator.max_run, "seconds")
print("Source directory:", source_dir)
print("Training job submitted: NO")
print("GPU launched: NO")

/home/sagemaker-user/medquad-envs/rag-env/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: PyTorch is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
PyTorch is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SageMaker Estimator: CONFIGURED
Instance: ml.g5.xlarge
Instance count: 1
Maximum runtime: 1800 seconds
Source directory: /home/sagemaker-user/medical_chatbot/deployment/medgemma
Training job submitted: NO
GPU launched: NO


In [32]:
from sagemaker.inputs import TrainingInput

experiment_input = TrainingInput(
    s3_data=(
        "s3://medquad-ai-cleaned/"
        "medgemma/experiments/mq019/input/"
    ),
    input_mode="File"
)

estimator.set_hyperparameters(
    config_file=(
        "/opt/ml/input/data/experiments/"
        "experiment_config.json"
    ),
    output_dir="/opt/ml/model"
)

print("SageMaker experiment input: CONFIGURED")
print("S3 input:", experiment_input.config["DataSource"]["S3DataSource"]["S3Uri"])
print("Hyperparameters:", estimator.hyperparameters())
print("Training job submitted: NO")
print("GPU launched: NO")

SageMaker experiment input: CONFIGURED
S3 input: s3://medquad-ai-cleaned/medgemma/experiments/mq019/input/
Hyperparameters: {'config_file': '"\\"/opt/ml/input/data/experiments/experiment_config.json\\""', 'output_dir': '"\\"/opt/ml/model\\""'}
Training job submitted: NO
GPU launched: NO


In [33]:
from pathlib import Path
import json

# Paths inside the future SageMaker container
container_input_dir = Path(
    "/opt/ml/input/data/experiments"
)

container_config = (
    container_input_dir / "experiment_config.json"
)

container_output_dir = Path("/opt/ml/model")

# Verify the configuration locally without launching SageMaker
local_config = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/medgemma_evaluation/"
    "experiment_config.json"
)

config = json.loads(
    local_config.read_text(encoding="utf-8")
)

assert len(config["experiments"]) == 2

for experiment in config["experiments"]:
    assert (
        local_config.parent / experiment["prompt_file"]
    ).is_file()

print("SageMaker input channel: experiments")
print("Container config path:", container_config)
print("Container output path:", container_output_dir)
print("Configured experiments:", len(config["experiments"]))
print("Local input validation: PASSED")
print("Training job submitted: NO")
print("GPU launched: NO")

SageMaker input channel: experiments
Container config path: /opt/ml/input/data/experiments/experiment_config.json
Container output path: /opt/ml/model
Configured experiments: 2
Local input validation: PASSED
Training job submitted: NO
GPU launched: NO


In [34]:
from pathlib import Path

deployment_dir = Path(
    "/home/sagemaker-user/medical_chatbot/deployment/medgemma"
)

assert (deployment_dir / "medgemma_job.py").is_file()
assert (deployment_dir / "requirements.txt").is_file()

assert estimator.instance_type == "ml.g5.xlarge"
assert estimator.instance_count == 1
assert estimator.max_run == 1800

hyperparameters = estimator.hyperparameters()

assert "config_file" in hyperparameters
assert "output_dir" in hyperparameters

assert experiment_input.config["DataSource"]["S3DataSource"]["S3Uri"] == (
    "s3://medquad-ai-cleaned/"
    "medgemma/experiments/mq019/input/"
)

print("=== SageMaker Preflight Validation ===")
print("Deployment directory: VERIFIED")
print("GPU instance:", estimator.instance_type)
print("Instance count:", estimator.instance_count)
print("Maximum runtime:", estimator.max_run, "seconds")
print("Experiment input: VERIFIED")
print("Hyperparameters: VERIFIED")
print("Training job submitted: NO")
print("GPU launched: NO")

=== SageMaker Preflight Validation ===
Deployment directory: VERIFIED
GPU instance: ml.g5.xlarge
Instance count: 1
Maximum runtime: 1800 seconds
Experiment input: VERIFIED
Hyperparameters: VERIFIED
Training job submitted: NO
GPU launched: NO


In [35]:
import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "index",
        "versions",
        "transformers"
    ],
    capture_output=True,
    text=True
)

print("Package index accessible:", result.returncode == 0)

# Check that our selected version is published
output = result.stdout + result.stderr

print(
    "Transformers 4.57.6 available:",
    "4.57.6" in output
)

# Check the model-loading class in our current environment
from transformers import AutoModelForImageTextToText

print("AutoModelForImageTextToText: AVAILABLE")
print("No model downloaded.")
print("No GPU launched.")

Package index accessible: True
Transformers 4.57.6 available: True
AutoModelForImageTextToText: AVAILABLE
No model downloaded.
No GPU launched.


In [36]:
from transformers import AutoConfig
from huggingface_hub import get_token

model_id = "google/medgemma-4b-it"

config = AutoConfig.from_pretrained(
    model_id,
    token=get_token()
)

print("=== MedGemma Model Configuration ===")
print("Model type:", config.model_type)
print("Architecture:", config.architectures)
print("Transformers version:", config.transformers_version)

print("\nNo model weights downloaded.")
print("No GPU launched.")

=== MedGemma Model Configuration ===
Model type: gemma3
Architecture: ['Gemma3ForConditionalGeneration']
Transformers version: 4.54.0.dev0

No model weights downloaded.
No GPU launched.


In [37]:
from transformers import (
    Gemma3ForConditionalGeneration,
    AutoModelForImageTextToText,
)
import transformers

print("Current Transformers:", transformers.__version__)
print("Gemma 3 architecture: AVAILABLE")
print("Gemma 3 class:", Gemma3ForConditionalGeneration.__name__)
print("Auto model class:", AutoModelForImageTextToText.__name__)

print("\nPlanned GPU Transformers version: 4.57.6")
print("No model weights downloaded.")
print("No GPU launched.")

Current Transformers: 5.20.1
Gemma 3 architecture: AVAILABLE
Gemma 3 class: Gemma3ForConditionalGeneration
Auto model class: AutoModelForImageTextToText

Planned GPU Transformers version: 4.57.6
No model weights downloaded.
No GPU launched.


In [38]:
import json
from pathlib import Path

root = Path("/home/sagemaker-user/medical_chatbot")
folder = root / "data/processed/medgemma_evaluation"

config = json.loads(
    (folder / "experiment_config.json").read_text(
        encoding="utf-8"
    )
)

assert config["model_id"] == "google/medgemma-4b-it"
assert len(config["experiments"]) == 2

output_names = []

for experiment in config["experiments"]:
    prompt_file = folder / experiment["prompt_file"]

    assert prompt_file.is_file(), (
        f"Missing prompt: {prompt_file}"
    )
    assert prompt_file.stat().st_size > 0

    output_names.append(experiment["output_file"])

    print(
        experiment["name"],
        "| Evidence:",
        experiment["evidence_count"],
        "| Prompt characters:",
        len(prompt_file.read_text(encoding="utf-8"))
    )

assert len(set(output_names)) == 2

print("\nExperiment configuration: VALID")
print("Both prompt files: VERIFIED")
print("Output filenames: UNIQUE")
print("GPU launched: NO")

baseline_rag | Evidence: 5 | Prompt characters: 3637
filtered_rag | Evidence: 1 | Prompt characters: 1226

Experiment configuration: VALID
Both prompt files: VERIFIED
Output filenames: UNIQUE
GPU launched: NO


In [39]:
expected_output = (
    "s3://medquad-ai-cleaned/"
    "medgemma/experiments/mq019/output/"
)

assert estimator.instance_type == "ml.g5.xlarge"
assert estimator.instance_count == 1
assert estimator.max_run == 1800
assert estimator.output_path == expected_output

print("=== SageMaker Execution Settings ===")
print("Instance:", estimator.instance_type)
print("Instance count:", estimator.instance_count)
print("Maximum runtime:", estimator.max_run, "seconds")
print("S3 output:", estimator.output_path)
print("Training job submitted: NO")
print("GPU launched: NO")

=== SageMaker Execution Settings ===
Instance: ml.g5.xlarge
Instance count: 1
Maximum runtime: 1800 seconds
S3 output: s3://medquad-ai-cleaned/medgemma/experiments/mq019/output/
Training job submitted: NO
GPU launched: NO


In [40]:
hyperparameters = estimator.hyperparameters()

input_config = experiment_input.config
input_uri = input_config["DataSource"]["S3DataSource"]["S3Uri"]

expected_input = (
    "s3://medquad-ai-cleaned/"
    "medgemma/experiments/mq019/input/"
)

expected_config = (
    "/opt/ml/input/data/experiments/"
    "experiment_config.json"
)

expected_output = "/opt/ml/model"

print("=== SageMaker Path Verification ===")
print("Input channel:", input_uri)
print("Config argument:", hyperparameters["config_file"])
print("Output argument:", hyperparameters["output_dir"])

assert input_uri == expected_input
assert expected_config in hyperparameters["config_file"]
assert expected_output in hyperparameters["output_dir"]

print("\nInput path: VERIFIED")
print("Output path: VERIFIED")
print("Training job submitted: NO")
print("GPU launched: NO")

=== SageMaker Path Verification ===
Input channel: s3://medquad-ai-cleaned/medgemma/experiments/mq019/input/
Config argument: "\"/opt/ml/input/data/experiments/experiment_config.json\""
Output argument: "\"/opt/ml/model\""

Input path: VERIFIED
Output path: VERIFIED
Training job submitted: NO
GPU launched: NO


In [41]:
import json

hyperparameters = estimator.hyperparameters()

for name in ["config_file", "output_dir"]:
    raw = hyperparameters[name]
    decoded = json.loads(raw)

    print(f"\n{name}")
    print("Raw:    ", repr(raw))
    print("Decoded:", repr(decoded))
    print("Type:   ", type(decoded).__name__)

print("\nTraining job submitted: NO")
print("GPU launched: NO")


config_file
Raw:     '"\\"/opt/ml/input/data/experiments/experiment_config.json\\""'
Decoded: '"/opt/ml/input/data/experiments/experiment_config.json"'
Type:    str

output_dir
Raw:     '"\\"/opt/ml/model\\""'
Decoded: '"/opt/ml/model"'
Type:    str

Training job submitted: NO
GPU launched: NO


In [42]:
import json

print("=== SageMaker Hyperparameter Inspection ===")

for name in ["config_file", "output_dir"]:
    stored = estimator._hyperparameters[name]
    serialized = estimator.hyperparameters()[name]

    print(f"\n{name}")
    print("Stored value:", repr(stored))
    print("Stored type:", type(stored).__name__)
    print("Serialized value:", repr(serialized))

    # Decode for inspection only.
    value = serialized

    for level in range(1, 4):
        try:
            value = json.loads(value)
            print(f"Decode {level}:", repr(value))
        except (json.JSONDecodeError, TypeError):
            break

print("\nNo configuration changes made.")
print("Training job submitted: NO")
print("GPU launched: NO")

=== SageMaker Hyperparameter Inspection ===

config_file
Stored value: '"/opt/ml/input/data/experiments/experiment_config.json"'
Stored type: str
Serialized value: '"\\"/opt/ml/input/data/experiments/experiment_config.json\\""'
Decode 1: '"/opt/ml/input/data/experiments/experiment_config.json"'
Decode 2: '/opt/ml/input/data/experiments/experiment_config.json'

output_dir
Stored value: '"/opt/ml/model"'
Stored type: str
Serialized value: '"\\"/opt/ml/model\\""'
Decode 1: '"/opt/ml/model"'
Decode 2: '/opt/ml/model'

No configuration changes made.
Training job submitted: NO
GPU launched: NO


In [43]:
import json

# Reset only the two affected hyperparameters.
estimator._hyperparameters["config_file"] = (
    "/opt/ml/input/data/experiments/experiment_config.json"
)

estimator._hyperparameters["output_dir"] = "/opt/ml/model"

expected = {
    "config_file": "/opt/ml/input/data/experiments/experiment_config.json",
    "output_dir": "/opt/ml/model",
}

for name, path in expected.items():
    stored = estimator._hyperparameters[name]
    serialized = estimator.hyperparameters()[name]

    assert stored == path
    assert json.loads(serialized) == path

    print(f"{name}: VERIFIED")
    print("  Stored:", repr(stored))
    print("  Serialized:", repr(serialized))

print("\nHyperparameter configuration: CORRECTED")
print("Training job submitted: NO")
print("GPU launched: NO")

config_file: VERIFIED
  Stored: '/opt/ml/input/data/experiments/experiment_config.json'
  Serialized: '"/opt/ml/input/data/experiments/experiment_config.json"'
output_dir: VERIFIED
  Stored: '/opt/ml/model'
  Serialized: '"/opt/ml/model"'

Hyperparameter configuration: CORRECTED
Training job submitted: NO
GPU launched: NO


In [44]:
print("=== SageMaker Network Configuration ===")

subnets = getattr(estimator, "subnets", None)
security_groups = getattr(estimator, "security_group_ids", None)
network_isolation = getattr(
    estimator,
    "enable_network_isolation",
    None
)

print("VPC subnets:", subnets)
print("Security groups:", security_groups)
print("Network isolation:", network_isolation)

if network_isolation is True:
    print("WARNING: Network isolation is enabled.")
else:
    print("Network isolation is not explicitly enabled.")

if subnets:
    print("VPC configuration: CUSTOM VPC")
else:
    print("VPC configuration: No custom subnets specified")

print("\nTraining job submitted: NO")
print("GPU launched: NO")

=== SageMaker Network Configuration ===
VPC subnets: None
Security groups: None
Network isolation: <bound method EstimatorBase.enable_network_isolation of <sagemaker.pytorch.estimator.PyTorch object at 0x7f8d4b4644d0>>
Network isolation is not explicitly enabled.
VPC configuration: No custom subnets specified

Training job submitted: NO
GPU launched: NO


In [45]:
import boto3
from botocore.exceptions import ClientError

ec2 = boto3.client("ec2", region_name="us-east-1")

try:
    response = ec2.describe_instance_type_offerings(
        LocationType="region",
        Filters=[
            {
                "Name": "instance-type",
                "Values": ["g5.xlarge", "g4dn.xlarge"]
            }
        ]
    )

    print("GPU instance types offered in region:")
    for item in response["InstanceTypeOfferings"]:
        print(item["InstanceType"])

except ClientError as error:
    print("AWS error:", error.response["Error"]["Code"])
    print(error.response["Error"]["Message"])

GPU instance types offered in region:
g4dn.xlarge
g5.xlarge
